# Phase 3: Predictive Modeling

Benchmark models → Optuna hyperparameter tuning → calibrated ensemble → SHAP explanations.

**Prerequisite**: run `02_feature_engineering.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_recall_curve
import shap

## 1. Load Data

## Optional: Load Pre-trained Model from Kaggle

If `../models/calibrated_model.pkl` already exists, run the cell below to skip all training and jump straight to evaluation and SHAP. Otherwise continue from **Section 1**.

In [ ]:
import os

PRETRAINED = os.path.exists('../models/calibrated_model.pkl')

if PRETRAINED:
    calibrated     = joblib.load('../models/calibrated_model.pkl')
    ensemble       = joblib.load('../models/ensemble_model.pkl')
    best_threshold_c = joblib.load('../models/threshold.pkl')
    feature_cols   = joblib.load('../models/feature_columns.pkl')

    # Load test set for evaluation (requires 02 to have been run)
    if os.path.exists('../data/X_test_raw.npy'):
        X_test_raw   = np.load('../data/X_test_raw.npy')
        y_test       = np.load('../data/y_test.npy')
        X_test_raw_df = pd.DataFrame(X_test_raw, columns=feature_cols)
        X_train_raw  = np.load('../data/X_train_raw.npy')
        y_train_raw  = np.load('../data/y_train_raw.npy')
        y_prob_c = calibrated.predict_proba(X_test_raw)[:, 1]
        y_final  = (y_prob_c >= best_threshold_c).astype(int)
        from sklearn.metrics import classification_report, roc_auc_score
        print(classification_report(y_test, y_final))
        print('ROC-AUC:', roc_auc_score(y_test, y_prob_c))
    else:
        print('Pre-trained model loaded. Run 02_feature_engineering first to get test splits for evaluation.')
else:
    print('No pre-trained model found — run sections 1–6 below to train from scratch.')

In [ ]:
X_train = np.load('../data/X_train.npy')       # SMOTE-resampled, scaled
X_test  = np.load('../data/X_test.npy')        # scaled
y_train = np.load('../data/y_train.npy')
y_test  = np.load('../data/y_test.npy')

X_train_raw = np.load('../data/X_train_raw.npy')  # unscaled, original class balance
X_test_raw  = np.load('../data/X_test_raw.npy')
y_train_raw = np.load('../data/y_train_raw.npy')

feature_cols = joblib.load('../models/feature_columns.pkl')
X_test_df = pd.DataFrame(X_test, columns=feature_cols)
X_test_raw_df = pd.DataFrame(X_test_raw, columns=feature_cols)

print(f'Train (SMOTE): {X_train.shape}, Test: {X_test.shape}')

## 2. Baseline Model Comparison

In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42),
    'XGBoost':             XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
    'LDA':                 LinearDiscriminantAnalysis(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'SVM':                 SVC(probability=True),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':         GaussianNB(),
    'LightGBM':            LGBMClassifier(random_state=42, verbosity=-1),
    'CatBoost':            CatBoostClassifier(random_state=42, verbose=0),
}

results = []
for name, model in baseline_models.items():
    # Tree models and LDA use unscaled data; linear models use SMOTE+scaled
    if name in ['Random Forest', 'XGBoost', 'Gradient Boosting', 'Decision Tree', 'LightGBM', 'CatBoost']:
        model.fit(X_train_raw, y_train_raw)
        prob = model.predict_proba(X_test_raw)[:, 1]
    else:
        model.fit(X_train, y_train)
        prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, prob)
    results.append({'Model': name, 'ROC-AUC': round(auc, 4)})

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(results_df.to_string(index=False))

### Baseline Results Summary

| Model | ROC-AUC |
|---|---|
| Logistic Regression | 0.846 |
| Gradient Boosting | 0.847 |
| LDA | 0.843 |
| Random Forest | 0.830 |
| Naive Bayes | 0.832 |
| XGBoost | 0.820 |
| SVM | 0.800 |
| KNN | 0.782 |
| Decision Tree | 0.656 |

Top 3 (LR, GBM, LDA) form the ensemble.

## 3. Hyperparameter Tuning with Optuna

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# --- Gradient Boosting ---
def objective_gb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
    }
    m = GradientBoostingClassifier(**params, random_state=42)
    return cross_val_score(m, X_train_raw, y_train_raw, cv=3, scoring='roc_auc').mean()

study_gb = optuna.create_study(direction='maximize')
study_gb.optimize(objective_gb, n_trials=40)
print('Best GB params:', study_gb.best_params)
print('Best ROC-AUC:', study_gb.best_value)

In [ ]:
# --- XGBoost ---
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    m = XGBClassifier(**params, random_state=42, eval_metric='logloss', verbosity=0)
    return cross_val_score(m, X_train_raw, y_train_raw, cv=3, scoring='roc_auc').mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=40)
print('Best XGB params:', study_xgb.best_params)
print('Best ROC-AUC:', study_xgb.best_value)

In [ ]:
# --- Logistic Regression ---
def objective_lr(trial):
    C = trial.suggest_float('C', 1e-3, 10, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])
    m = LogisticRegression(C=C, solver=solver, max_iter=1000)
    return cross_val_score(m, X_train, y_train, cv=3, scoring='roc_auc').mean()

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=30)
print('Best LR params:', study_lr.best_params)
print('Best ROC-AUC:', study_lr.best_value)

In [ ]:
# --- LDA ---
def objective_lda(trial):
    solver = trial.suggest_categorical('solver', ['svd', 'lsqr'])
    shrinkage = None if solver == 'svd' else trial.suggest_float('shrinkage', 0.0, 1.0)
    m = LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage)
    return cross_val_score(m, X_train, y_train, cv=3, scoring='roc_auc').mean()

study_lda = optuna.create_study(direction='maximize')
study_lda.optimize(objective_lda, n_trials=20)
print('Best LDA params:', study_lda.best_params)
print('Best ROC-AUC:', study_lda.best_value)

## 4. Build Tuned Ensemble

In [ ]:
# LR with SMOTE inside pipeline so it applies during CV
lr_tuned = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(**{k: v for k, v in study_lr.best_params.items()}, max_iter=1000))
])

gb_tuned  = GradientBoostingClassifier(**study_gb.best_params, random_state=42)
lda_tuned = LinearDiscriminantAnalysis(**{k: v for k, v in study_lda.best_params.items() if k != 'shrinkage' or study_lda.best_params.get('solver') != 'svd'})
xgb_tuned = XGBClassifier(**study_xgb.best_params, random_state=42, eval_metric='logloss', verbosity=0)

ensemble = VotingClassifier(
    estimators=[('lr', lr_tuned), ('gb', gb_tuned), ('xgb', xgb_tuned)],
    voting='soft'
)
ensemble.fit(X_train_raw, y_train_raw)

## 5. Threshold Tuning

In [ ]:
y_proba = ensemble.predict_proba(X_test_raw)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
print(f'Best threshold (max F1): {best_threshold:.4f}')

y_pred = (y_proba >= best_threshold).astype(int)
print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))

## 6. Calibration

In [ ]:
calibrated = CalibratedClassifierCV(ensemble, method='isotonic', cv=3)
calibrated.fit(X_train_raw, y_train_raw)

y_prob_c = calibrated.predict_proba(X_test_raw)[:, 1]

precision_c, recall_c, thresholds_c = precision_recall_curve(y_test, y_prob_c)
f1_c = 2 * (precision_c * recall_c) / (precision_c + recall_c + 1e-9)
best_threshold_c = thresholds_c[np.argmax(f1_c[:-1])]
print(f'Best threshold (calibrated): {best_threshold_c:.4f}')

y_final = (y_prob_c >= best_threshold_c).astype(int)
print(classification_report(y_test, y_final))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_c))

## 7. SHAP — XGBoost (tuned)

In [ ]:
xgb_tuned.fit(X_train_raw, y_train_raw)
explainer_xgb = shap.TreeExplainer(xgb_tuned)
shap_vals_xgb = explainer_xgb(X_test_raw_df)

shap.summary_plot(shap_vals_xgb, X_test_raw_df)
shap.plots.bar(shap_vals_xgb)

## 8. SHAP — Gradient Boosting (tuned)

In [ ]:
gb_tuned.fit(X_train_raw, y_train_raw)
explainer_gb = shap.TreeExplainer(gb_tuned)
shap_vals_gb = explainer_gb(X_test_raw_df)

shap.summary_plot(shap_vals_gb, X_test_raw_df)
shap.plots.bar(shap_vals_gb)

## 9. Feature Importance (XGBoost)

In [ ]:
feat_imp = pd.Series(xgb_tuned.feature_importances_, index=feature_cols).sort_values()
feat_imp.tail(15).plot(kind='barh', figsize=(8, 6))
plt.title('XGBoost Feature Importance (top 15)')
plt.tight_layout()
plt.show()
print(feat_imp.tail(15))

## 10. Save Model Artefacts

In [ ]:
joblib.dump(ensemble,       '../models/ensemble_model.pkl')
joblib.dump(calibrated,     '../models/calibrated_model.pkl')
joblib.dump(best_threshold_c, '../models/threshold.pkl')
print('Saved: ensemble_model.pkl, calibrated_model.pkl, threshold.pkl')